In [4]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ============================================================
# 1. LOAD DATASET
# ============================================================

BASE_DIR = Path.cwd().parent

DATA_PATH = BASE_DIR / "data" / "customer_churn.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())


# ============================================================
# 2. BASIC CLEANING
# ============================================================

# Convert TotalCharges to numeric
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    )

# Fill missing numeric values
if "TotalCharges" in df.columns:
    df["TotalCharges"] = df["TotalCharges"].fillna(
        df["TotalCharges"].median()
    )


# ============================================================
# 3. REMOVE ID COLUMN
# ============================================================

if "customerID" in df.columns:
    df = df.drop("customerID", axis=1)


# ============================================================
# 4. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop("Churn", axis=1)
y = df["Churn"]


# ============================================================
# 5. ENCODE TARGET
# ============================================================

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

print("\nTarget Encoding:")
print(dict(zip(
    target_encoder.classes_,
    target_encoder.transform(target_encoder.classes_)
)))


# ============================================================
# 6. ENCODE CATEGORICAL FEATURES
# ============================================================

label_encoders = {}

for col in X.select_dtypes(include="object").columns:

    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(X[col].astype(str))

    label_encoders[col] = encoder


print("\nFeature Encoding Completed.")


# ============================================================
# 7. FEATURE COLUMNS
# ============================================================

feature_columns = X.columns.tolist()

print("\nFinal Feature Columns:")
print(feature_columns)

print("\nNumber of Features:", len(feature_columns))


# ============================================================
# 8. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\nTraining Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


# ============================================================
# 9. RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)


# ============================================================
# 10. MODEL EVALUATION
# ============================================================

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n====================================")
print("MODEL PERFORMANCE")
print("====================================")

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# ============================================================
# 11. SAVE MODEL FILES
# ============================================================

MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(exist_ok=True)


# Save model
joblib.dump(
    model,
    MODEL_DIR / "churn_model.pkl"
)


# Save feature columns
joblib.dump(
    feature_columns,
    MODEL_DIR / "feature_columns.pkl"
)


# Save feature encoders
joblib.dump(
    label_encoders,
    MODEL_DIR / "label_encoders.pkl"
)


# Save target encoder
joblib.dump(
    target_encoder,
    MODEL_DIR / "target_encoder.pkl"
)


print("\n====================================")
print("EVERYTHING SAVED SUCCESSFULLY")
print("====================================")

print("churn_model.pkl       ✅")
print("feature_columns.pkl   ✅")
print("label_encoders.pkl    ✅")
print("target_encoder.pkl    ✅")

Dataset Shape: (7043, 21)
Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Target Encoding:
{'No': np.int64(0), 'Yes': np.int64(1)}

Feature Encoding Completed.

Final Feature Columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Number of Features: 19

Training Shape: (5634, 19)
Testing Shape: (1409, 19)


C:\Users\kaviy\AppData\Local\Temp\ipykernel_23864\21752378.py:80: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:



MODEL PERFORMANCE
Accuracy: 77.36 %

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1035
           1       0.56      0.64      0.60       374

    accuracy                           0.77      1409
   macro avg       0.71      0.73      0.72      1409
weighted avg       0.78      0.77      0.78      1409


EVERYTHING SAVED SUCCESSFULLY
churn_model.pkl       ✅
feature_columns.pkl   ✅
label_encoders.pkl    ✅
target_encoder.pkl    ✅
